In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [2]:
import torch
import torchaudio
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample

In [3]:
dataset = load_from_disk('/scratch/asudupe/datasets/VoiceAssistant-400K_eu/dataset_with_token_paths/')

In [ ]:
speech, sr = torchaudio.load(os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['train'][10]['question_audio']))
Audio(data=np.array(speech), rate=sr)

In [15]:
dataset['train'][10]['answer']

'Txakur bat apaintzeak hainbat urrats dakartza. Has zaitez zure txakurraren ilea eskuilatzen, korapiloak eta ile solteak kentzeko. Ondoren, eman bainu bat zure txakurrari, txakurren ile-apainketarako xanpu egokia erabiliz, eta ziurtatu ondo garbitzen duzula hondakinik ez uzteko. Bainuaren ondoren, lehortu zure txakurra eskuoihal batekin edo animalientzako lehorgailu batekin. Moztu zure txakurraren azazkalak kontu handiz, azazkalaren erroa ukitu gabe. Azkenik, garbitu zure txakurraren belarriak albaitariak gomendatutako belarri-garbitzaile batekin eta garbitu hortzak txakurrentzako hortzetako pastarekin. Izan beti leuna eta eskaini sariak prozesuan zehar zure txakurra lasai eta eroso mantentzeko.'

In [6]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [13]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [4]:
# model_path = 'saves/13834/checkpoint-24000'
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage1/best/checkpoint-20976"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = False
mel_size = 128
conv_mode = 'llama_3'

In [5]:
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [31]:
# speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1010]['question_audio'])
speech_file = "audioak/Recording_5.mp3"
speech = whisper.load_audio(speech_file)
# speech = Resample(orig_freq=sr, new_freq=16000)(speech)
# speech = speech.squeeze()
Audio(data=np.array(speech), rate = 16000)

In [32]:
qs = "<speech>\nPlease directly answer the questions in the user's speech."
# audio = dataset['train'][20]['question_audio']
# speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = torch.stack((input_ids, input_ids), dim=0)
speech_tensors = torch.stack((speech_tensor, speech_tensor), dim=0)
speech_lengths = torch.stack((speech_length, speech_length), dim=0)


In [33]:
temperature = 0.0
top_p = None
num_beams = 1
max_new_tokens = 256

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
        
    )
output_ids = outputs

outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
outputs

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


'Hori zoragarria da! Hainbat aukera dituzu zure proiektua aurrera eramateko. Webgune bat sor dezakezu zure artisautza erakusteko, blog bat hasi zure esperientziak eta tutorialak partekatzeko, edo online denda bat sortu zure produktuak saltzeko. Sare sozialetan ere presentzia indartsua eraiki dezakezu zure lana erakusteko eta zure publikoarekin harremanetan jartzeko. Beste artisau batzuekin elkarlanean aritzea eta komunitateetan edo foroetan parte hartzea ere lagungarria izan daiteke hazkunde eta aukera berrietarako.'